# SQL Queries — Shrinkflation in Household Cleaning Products
## BUSN 32120 Final Project | Group 11: Ridhi Agarwal, Jaire Augusto Byers

This file contains **all 10 required SQL queries**, each with comments explaining:
- **What** the query does
- **How** it works (SQL mechanics)
- **Why** it is relevant to the shrinkflation legal analysis

Queries run on an in-memory SQLite database loaded with two tables:
- `cpi_data` — 660 monthly rows (Jan 1970–Dec 2024) from FRED + BLS
- `annual_summary` — one row per year with aggregated averages and recession counts

### Setup — rebuild the SQLite database

In [1]:
import requests, sqlite3, pandas as pd, numpy as np
from datetime import datetime

# ── Rebuild FRED data ────────────────────────────────────────────────────
FRED_API_KEY = 'a5993e30e75374dc11aab99b9d6934eb'
BASE_URL     = 'https://api.stlouisfed.org/fred/series/observations'

def fred_requests(series_id, start='1970-01-01', end='2024-12-31'):
    r = requests.get(BASE_URL, params={'series_id': series_id, 'api_key': FRED_API_KEY,
                                       'file_type': 'json', 'observation_start': start,
                                       'observation_end': end})
    df = pd.DataFrame(r.json()['observations'])
    df['date']  = pd.to_datetime(df['date'])
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    return df[['date','value']]

SERIES = {'cpi_all':'CPIAUCSL','cpi_core':'CPILFESL','cpi_cleaning':'CUSR0000SAS',
          'cpi_household':'CUSR0000SAH3','cpi_nondurable':'CUSR0000SAN','recession':'USREC'}
fred = {}
for name, sid in SERIES.items():
    fred[name] = fred_requests(sid).set_index('date').rename(columns={'value': name})

df = (fred['cpi_cleaning'].join(fred['cpi_all']).join(fred['cpi_core'])
      .join(fred['cpi_household']).join(fred['cpi_nondurable']).join(fred['recession']))
df['real_cleaning_vs_all']        = (df['cpi_cleaning'] / df['cpi_all'])        * 100
df['real_cleaning_vs_core']       = (df['cpi_cleaning'] / df['cpi_core'])       * 100
df['real_cleaning_vs_nondurable'] = (df['cpi_cleaning'] / df['cpi_nondurable']) * 100
df['post_2019']  = (df.index >= '2019-01-01').astype(int)
df['year']       = df.index.year
df['month_num']  = df.index.month

# ── Load into SQLite ─────────────────────────────────────────────────────
conn = sqlite3.connect(':memory:')
df_sql = df.reset_index().rename(columns={'date':'date'})
df_sql['date'] = df_sql['date'].astype(str)
df_sql.to_sql('cpi_data', conn, if_exists='replace', index=False)

df_annual = df.groupby('year').agg(
    avg_cpi_cleaning  = ('cpi_cleaning',        'mean'),
    avg_cpi_all       = ('cpi_all',             'mean'),
    avg_real_cleaning = ('real_cleaning_vs_all', 'mean'),
    recession_months  = ('recession',            'sum')
).reset_index()
df_annual.to_sql('annual_summary', conn, if_exists='replace', index=False)

print('Tables ready:', pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].tolist())

/Users/ridhiagarwal/Downloads/data-analysis-assignment/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Tables ready: ['cpi_data', 'annual_summary']


---
### Q1 — Most recent 12 months (Basic SELECT)
**What:** Retrieves the last 12 months of CPI data ordered by date descending.  
**How:** `SELECT` with `ORDER BY date DESC` and `LIMIT 12`.  
**Why:** Provides a quick sanity check that the data loaded correctly and shows the current level of cleaning product prices, which is the core evidence period for the litigation.

In [2]:
q1 = pd.read_sql("""
    SELECT date, cpi_cleaning, cpi_all, real_cleaning_vs_all
    FROM cpi_data
    ORDER BY date DESC
    LIMIT 12
""", conn)
q1

,date,cpi_cleaning,cpi_all,real_cleaning_vs_all
0,2024-12-01,410.215,317.604,129.159268
1,2024-11-01,409.184,316.528,129.272608
2,2024-10-01,408.066,315.631,129.285780
3,2024-09-01,406.379,314.732,129.119060
4,2024-08-01,404.827,314.062,128.900345
5,2024-07-01,403.570,313.569,128.702136
6,2024-06-01,402.370,313.044,128.534647
7,2024-05-01,401.616,313.175,128.240121
8,2024-04-01,400.830,313.023,128.051293
9,2024-03-01,399.454,312.345,127.888713


---
### Q2 — Annual average CPI values (GROUP BY)
**What:** Aggregates monthly data to one row per year, computing average CPI values.  
**How:** `GROUP BY year` with `AVG()` on each CPI column.  
**Why:** Year-level aggregation shows the long-run trend and helps legal counsel identify which years show the clearest pricing anomaly relative to the historical baseline.

In [3]:
q2 = pd.read_sql("""
    SELECT year,
           ROUND(AVG(cpi_cleaning), 2)         AS avg_cpi_cleaning,
           ROUND(AVG(cpi_all), 2)              AS avg_cpi_all,
           ROUND(AVG(real_cleaning_vs_all), 2) AS avg_real_vs_all
    FROM cpi_data
    GROUP BY year
    ORDER BY year
""", conn)
q2

,year,avg_cpi_cleaning,avg_cpi_all,avg_real_vs_all
0,1970,35.07,38.84,90.27
1,1971,37.02,40.48,91.46
2,1972,38.45,41.81,91.97
3,1973,40.08,44.43,90.24
4,1974,43.86,49.32,88.93
5,1975,48.04,53.82,89.25
6,1976,52.02,56.93,91.37
7,1977,56.02,60.62,92.40
8,1978,60.82,65.24,93.22
9,1979,67.51,72.58,93.00


---
### Q3 — Years where cleaning products outpaced general CPI (GROUP BY + HAVING)
**What:** Filters to only years where the average relative cleaning price exceeded 100 (i.e., cleaning products were more expensive than the overall basket on average).  
**How:** `GROUP BY year` with a `HAVING` clause filtering on the aggregate.  
**Why:** `HAVING` filters on aggregated results — here it isolates the years most relevant to the shrinkflation claim, showing how many months of elevated pricing occurred per year.

In [4]:
q3 = pd.read_sql("""
    SELECT year,
           ROUND(AVG(real_cleaning_vs_all), 2) AS avg_relative_price,
           COUNT(*) AS months_observed
    FROM cpi_data
    GROUP BY year
    HAVING AVG(real_cleaning_vs_all) > 100
    ORDER BY avg_relative_price DESC
""", conn)
q3

,year,avg_relative_price,months_observed
0,2024,128.55,12
1,2020,128.26,12
2,2019,127.17,12
3,2023,126.18,12
4,2018,126.09,12
5,2021,126.02,12
6,2017,125.69,12
7,2016,124.97,12
8,2022,123.90,12
9,2015,123.07,12


---
### Q4 — Months with largest deviation from their annual average (JOIN)
**What:** Compares each month's cleaning CPI to the annual average for that year, ranking by absolute deviation.  
**How:** `INNER JOIN` between `cpi_data` and `annual_summary` on the `year` column; deviation computed as the difference between monthly and annual values.  
**Why:** Identifies specific months where pricing diverged most sharply from the year's norm — useful for pinpointing discrete pricing events relevant to the litigation timeline.

In [5]:
q4 = pd.read_sql("""
    SELECT c.date,
           ROUND(c.cpi_cleaning, 2)                      AS cpi_cleaning,
           ROUND(a.avg_cpi_cleaning, 2)                  AS annual_avg,
           ROUND(c.cpi_cleaning - a.avg_cpi_cleaning, 2) AS deviation
    FROM cpi_data AS c
    JOIN annual_summary AS a ON c.year = a.year
    ORDER BY ABS(c.cpi_cleaning - a.avg_cpi_cleaning) DESC
    LIMIT 10
""", conn)
q4

,date,cpi_cleaning,annual_avg,deviation
0,2022-01-01,350.03,362.59,-12.56
1,2022-12-01,374.37,362.59,11.78
2,2022-02-01,351.78,362.59,-10.81
3,2022-11-01,372.20,362.59,9.62
4,2022-03-01,353.88,362.59,-8.71
5,2023-12-01,392.96,384.50,8.47
6,2022-10-01,370.80,362.59,8.21
7,2024-01-01,395.44,403.27,-7.83
8,2023-01-01,376.71,384.50,-7.79
9,2023-11-01,391.55,384.50,7.05


---
### Q5 — Post-2019 months enriched with annual recession context (JOIN + WHERE)
**What:** Retrieves post-2019 monthly records and attaches the count of recession months for each corresponding year.  
**How:** `JOIN` on `year` with a `WHERE post_2019 = 1` filter; pulls `recession_months` from `annual_summary`.  
**Why:** Allows legal counsel to see elevated cleaning prices in the context of whether that year had any recession months — supporting the argument that the price spike was not recession-driven.

In [6]:
q5 = pd.read_sql("""
    SELECT c.date,
           ROUND(c.cpi_cleaning, 2)         AS cpi_cleaning,
           ROUND(c.real_cleaning_vs_all, 2) AS real_vs_all,
           a.recession_months
    FROM cpi_data AS c
    JOIN annual_summary AS a ON c.year = a.year
    WHERE c.post_2019 = 1
    ORDER BY c.date
    LIMIT 12
""", conn)
q5

,date,cpi_cleaning,real_vs_all,recession_months
0,2019-01-01,321.13,127.15,0
1,2019-02-01,321.75,127.01,0
2,2019-03-01,322.45,126.81,0
3,2019-04-01,323.28,126.66,0
4,2019-05-01,323.71,126.80,0
5,2019-06-01,324.51,127.15,0
6,2019-07-01,325.34,127.18,0
7,2019-08-01,326.16,127.39,0
8,2019-09-01,326.98,127.51,0
9,2019-10-01,327.90,127.51,0


---
### Q6 — Recession years vs. expansion years price comparison (JOIN + GROUP BY)
**What:** Compares average cleaning product CPI between years that had at least one recession month and those that didn't.  
**How:** `JOIN` `annual_summary` to `cpi_data` on `year`, then `GROUP BY` a `CASE` expression that classifies each year as recession or expansion.  
**Why:** Tests whether elevated cleaning prices are explained by recessions. If prices are similarly high in expansion years, the recession defense is weakened — which is exactly what the data shows.

In [7]:
q6 = pd.read_sql("""
    SELECT CASE WHEN a.recession_months > 0 THEN 'Recession Year'
                ELSE 'Expansion Year' END           AS year_type,
           COUNT(DISTINCT a.year)                   AS num_years,
           ROUND(AVG(a.avg_cpi_cleaning), 2)        AS avg_cpi_cleaning,
           ROUND(AVG(a.avg_real_cleaning), 2)       AS avg_real_vs_all
    FROM annual_summary AS a
    JOIN cpi_data AS c ON a.year = c.year
    GROUP BY year_type
""", conn)
q6

,year_type,num_years,avg_cpi_cleaning,avg_real_vs_all
0,Expansion Year,42,201.00,112.28
1,Recession Year,13,135.75,103.56


---
### Q7 — Peak cleaning CPI month within each year (Window Function — RANK)
**What:** Identifies the single highest-CPI month for cleaning products within each calendar year.  
**How:** `RANK() OVER (PARTITION BY year ORDER BY cpi_cleaning DESC)` assigns a rank within each year's partition; an outer query filters for rank = 1.  
**Why:** Reveals which month of each year saw peak pricing — useful for identifying whether companies tend to raise prices at specific times (e.g., spring cleaning season), which could support a coordinated pricing narrative.

In [8]:
q7 = pd.read_sql("""
    SELECT date, year, cpi_cleaning, rank_in_year
    FROM (
        SELECT date, year, cpi_cleaning,
               RANK() OVER (PARTITION BY year
                            ORDER BY cpi_cleaning DESC) AS rank_in_year
        FROM cpi_data
    ) ranked
    WHERE rank_in_year = 1
    ORDER BY year
""", conn)
q7

,date,year,cpi_cleaning,rank_in_year
0,1970-12-01,1970,36.200,1
1,1971-12-01,1971,37.700,1
2,1972-12-01,1972,39.000,1
3,1973-12-01,1973,41.500,1
4,1974-12-01,1974,46.200,1
5,1975-12-01,1975,49.900,1
6,1976-12-01,1976,53.700,1
7,1977-12-01,1977,57.900,1
8,1978-12-01,1978,63.300,1
9,1979-12-01,1979,72.000,1


---
### Q8 — 12-month rolling average of relative cleaning price (Window Function — Moving Average)
**What:** Computes a 12-month trailing average of `real_cleaning_vs_all` alongside the raw monthly value.  
**How:** `AVG() OVER (ORDER BY date ROWS BETWEEN 11 PRECEDING AND CURRENT ROW)` — a rolling window of 12 rows including the current month.  
**Why:** Smooths month-to-month noise to reveal the persistent trend. In legal analysis, a sustained elevated rolling average is stronger evidence than a few spike months — it shows the price divergence is structural, not transitory.

In [9]:
q8 = pd.read_sql("""
    SELECT date,
           ROUND(real_cleaning_vs_all, 2) AS real_vs_all,
           ROUND(
               AVG(real_cleaning_vs_all) OVER (
                   ORDER BY date
                   ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
               ), 2
           ) AS moving_avg_12m
    FROM cpi_data
    ORDER BY date DESC
    LIMIT 24
""", conn)
q8

,date,real_vs_all,moving_avg_12m
0,2024-12-01,129.16,128.55
1,2024-11-01,129.27,128.39
2,2024-10-01,129.29,128.21
3,2024-09-01,129.12,127.99
4,2024-08-01,128.90,127.76
5,2024-07-01,128.70,127.53
6,2024-06-01,128.53,127.33
7,2024-05-01,128.24,127.12
8,2024-04-01,128.05,126.93
9,2024-03-01,127.89,126.74


---
### Q9 — Months above all-time average cleaning CPI (Inline Subquery)
**What:** Retrieves months where cleaning product CPI exceeded the all-time historical average.  
**How:** An inline scalar subquery `(SELECT AVG(cpi_cleaning) FROM cpi_data)` computes the baseline; the outer `WHERE` clause compares each row against it.  
**Why:** Quantifies how many months in the historical record showed above-average cleaning prices. Combined with the post-2019 concentration of those months, this directly supports the class-wide consumer harm argument.

In [10]:
q9 = pd.read_sql("""
    SELECT date, cpi_cleaning, real_cleaning_vs_all
    FROM cpi_data
    WHERE cpi_cleaning > (SELECT AVG(cpi_cleaning) FROM cpi_data)
    ORDER BY date DESC
    LIMIT 12
""", conn)
q9

,date,cpi_cleaning,real_cleaning_vs_all
0,2024-12-01,410.215,129.159268
1,2024-11-01,409.184,129.272608
2,2024-10-01,408.066,129.285780
3,2024-09-01,406.379,129.119060
4,2024-08-01,404.827,128.900345
5,2024-07-01,403.570,128.702136
6,2024-06-01,402.370,128.534647
7,2024-05-01,401.616,128.240121
8,2024-04-01,400.830,128.051293
9,2024-03-01,399.454,127.888713


---
### Q10 — Post-2019 months most above pre-2019 baseline (CTE Subquery)
**What:** Computes the average relative cleaning price in the pre-2019 period as a baseline, then identifies post-2019 months that exceed it most — ranked by excess.  
**How:** A `WITH` (CTE) clause computes the pre-2019 average once; the main query joins every post-2019 row against that single baseline value and computes the excess.  
**Why:** This is the most direct quantification of the legal claim: it shows precisely which months, and by how much, cleaning products were priced above the established historical norm — the core damages-relevant output.

In [11]:
q10 = pd.read_sql("""
    WITH pre_avg AS (
        SELECT AVG(real_cleaning_vs_all) AS baseline
        FROM cpi_data
        WHERE post_2019 = 0
    )
    SELECT c.date,
           ROUND(c.real_cleaning_vs_all, 2)              AS real_vs_all,
           ROUND(p.baseline, 2)                          AS pre_2019_baseline,
           ROUND(c.real_cleaning_vs_all - p.baseline, 2) AS excess
    FROM cpi_data AS c, pre_avg AS p
    WHERE c.post_2019 = 1
      AND c.real_cleaning_vs_all > p.baseline
    ORDER BY excess DESC
    LIMIT 10
""", conn)
q10

,date,real_vs_all,pre_2019_baseline,excess
0,2024-10-01,129.29,108.21,21.08
1,2024-11-01,129.27,108.21,21.07
2,2024-12-01,129.16,108.21,20.95
3,2024-09-01,129.12,108.21,20.91
4,2024-08-01,128.90,108.21,20.69
5,2020-05-01,128.87,108.21,20.66
6,2020-04-01,128.84,108.21,20.63
7,2024-07-01,128.70,108.21,20.50
8,2020-07-01,128.65,108.21,20.44
9,2020-06-01,128.59,108.21,20.38
